In [21]:
import pandas as pd
import numpy as np
from libpysal.weights import full2W
from esda.getisord import G
from pyprojroot import here

In [22]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [23]:
crime = pd.read_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding='latin-1')
adj = pd.read_csv(INTERIM_DATA_DIR/'adj_combined.csv', encoding='latin-1', index_col=0, dtype=str)

In [24]:
crime["CVEGEO"] = crime["CVEGEO"].astype(str).str.zfill(5)
adj.index = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)

In [25]:
common_ids = sorted(set(adj.index) & set(crime["CVEGEO"]))
adj = adj.loc[common_ids, common_ids]

In [26]:
w = full2W(adj.values.astype(int), ids=common_ids)
w.transform = "r" 

print(f"True islands: {w.islands}") 

True islands: []


In [27]:
results = []
years = sorted(crime["year"].unique())

for year in years:
    yr_data = crime[crime["year"] == year].set_index("CVEGEO").reindex(common_ids)

    if yr_data["crime_count"].isna().any():
        missing = yr_data[yr_data["crime_count"].isna()].index.tolist()
        print(f"Year {year}: missing crime_count for {len(missing)} municipalities — check data")
        continue

    y = yr_data["crime_count"].values.astype(float)
    g_stats = G(y, w, permutations=999)

    results.append({
        "year": year,
        "observed_g": g_stats.G,
        "expected_g": g_stats.EG_sim,
        "p_sim": g_stats.p_sim,
        "p_norm": g_stats.p_norm,
        "z_sim": g_stats.z_sim,
    })

getis_ord_g = pd.DataFrame(results)
print(getis_ord_g)

    year  observed_g  expected_g  p_sim        p_norm      z_sim
0   2015    0.019756    0.002403  0.001  0.000000e+00  17.108485
1   2016    0.017348    0.002392  0.001  0.000000e+00  14.872374
2   2017    0.018072    0.002417  0.002  0.000000e+00   9.948483
3   2018    0.021476    0.002342  0.001  0.000000e+00  12.244820
4   2019    0.017639    0.002279  0.001  0.000000e+00  11.101199
5   2020    0.012861    0.002382  0.008  6.791554e-07   5.118824
6   2021    0.013572    0.002371  0.006  9.315170e-07   4.823598
7   2022    0.014124    0.002386  0.005  4.489725e-09   5.305122
8   2023    0.015765    0.002390  0.006  1.554312e-15   6.892858
9   2024    0.014880    0.002380  0.001  3.441691e-15   7.686213
10  2025    0.016545    0.002319  0.001  0.000000e+00  11.455079


In [28]:
getis_ord_g.to_csv(RESULTS_DATA_DIR/"raw_esda_results/yearly_getisord_g.csv", index=False)